# CardingForums — 02. Análisis estructural

En este notebook analizamos la **estructura** del foro — quién habla con quién, cómo se organizan las comunidades, qué patrones temporales hay, y qué términos dominan en cada subforo.

Todo lo que hacemos aquí es **sin IA generativa**: usamos algoritmos clásicos de análisis de redes, estadística y NLP. Esto es importante porque:
1. Son reproducibles y auditables
2. No requieren GPU ni APIs externas
3. Dan resultados que podemos validar matemáticamente

---

## 1. Setup y carga de datos limpios

Cargamos los datos procesados en el notebook 01. El flujo de notebooks es en cadena: cada uno produce outputs que el siguiente consume.

**Librería nueva: NetworkX**  
NetworkX es la librería estándar de Python para análisis de grafos (redes). Un grafo es una estructura de nodos y aristas — en nuestro caso, nodos = usuarios y aristas = participación compartida en threads.

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from pathlib import Path
from collections import Counter

from src.utils import RESULTS_DIR

sns.set_theme(style="darkgrid")

# Cargar los datos limpios del notebook 01
users      = pd.read_parquet(RESULTS_DIR / "01_users_clean.parquet")
posts      = pd.read_parquet(RESULTS_DIR / "01_posts_clean.parquet")
user_thread = pd.read_parquet(RESULTS_DIR / "01_user_thread.parquet")

print(f"Usuarios: {len(users):,}")
print(f"Posts:    {len(posts):,}")
print(f"Pares usuario-thread: {len(user_thread):,}")

## 2. Red de co-participación

### ¿Qué es una red de co-participación?

Dos usuarios están "conectados" si participaron en el mismo thread. La intuición es que si ambos escribieron en la misma conversación, hubo algún tipo de interacción (directa o indirecta).

El proceso:
1. Para cada thread, tomamos todos los usuarios que participaron
2. Creamos una arista entre cada par de usuarios de ese thread
3. El peso de la arista es el número de threads que comparten

**¿Por qué no usar @menciones directas?**  
Porque los datos de vBulletin no siempre capturan quién responde a quién directamente. La co-participación en threads es una señal más robusta y universal.

In [ ]:
# Construir la red de co-participación
# Para cada thread, generar pares de usuarios (combinaciones de 2)
from itertools import combinations

edge_counts = Counter()

for (forum, threadid), group in user_thread.groupby(["forum", "threadid"]):
    user_ids = group["userid"].dropna().astype(int).tolist()
    if len(user_ids) >= 2:
        for u1, u2 in combinations(sorted(set(user_ids)), 2):
            edge_counts[(u1, u2, forum)] += 1

# Convertir a DataFrame de aristas
edges_df = pd.DataFrame(
    [(u1, u2, f, w) for (u1, u2, f), w in edge_counts.items()],
    columns=["user_a", "user_b", "forum", "weight"]
)

print(f"Aristas (pares de usuarios co-presentes en un thread): {len(edges_df):,}")
print(f"Aristas con peso ≥ 2 (comparten al menos 2 threads): {(edges_df['weight'] >= 2).sum():,}")
edges_df.head()

### Construir el grafo con NetworkX

NetworkX puede construir un grafo directamente desde un DataFrame de aristas. Filtramos aristas de bajo peso (usuarios que comparten solo 1 thread) para quedarnos con conexiones más significativas.

**¿Qué es un grafo no dirigido?**  
En un grafo no dirigido, si A está conectado con B, entonces B está conectado con A. No hay sentido en la relación. Lo usamos aquí porque "co-participación" es simétrica: si yo estuve en el mismo thread que tú, tú estuviste en el mismo thread que yo.

In [ ]:
# Filtrar aristas débiles para que el grafo sea manejable
MIN_SHARED_THREADS = 2
strong_edges = edges_df[edges_df["weight"] >= MIN_SHARED_THREADS]

# Construir el grafo
G = nx.Graph()

# Añadir aristas con peso
for _, row in strong_edges.iterrows():
    if G.has_edge(row["user_a"], row["user_b"]):
        G[row["user_a"]][row["user_b"]]["weight"] += row["weight"]
    else:
        G.add_edge(row["user_a"], row["user_b"], weight=row["weight"])

print(f"Grafo construido:")
print(f"  Nodos (usuarios): {G.number_of_nodes():,}")
print(f"  Aristas (conexiones): {G.number_of_edges():,}")
print(f"  Densidad: {nx.density(G):.4f}")
print(f"  ¿Conectado? {nx.is_connected(G)}")

# Componente gigante (la parte más grande de la red conectada)
giant = max(nx.connected_components(G), key=len)
G_giant = G.subgraph(giant).copy()
print(f"  Componente gigante: {len(giant):,} nodos ({100*len(giant)/G.number_of_nodes():.1f}% del total)")

## 3. Métricas de centralidad

Las métricas de centralidad nos dicen quién es "importante" en la red, pero hay diferentes definiciones de importancia:

### ¿Qué es el degree centrality?

El **degree** de un nodo es simplemente el número de conexiones directas que tiene. Un usuario con degree alto conoce (co-participó) con muchos otros usuarios. Es la métrica más simple de "popularidad" en la red.

El degree *centrality* normaliza esto entre 0 y 1 dividiendo por el número máximo posible de conexiones.

### ¿Qué es la betweenness centrality?

Imagina que quieres ir del usuario A al usuario C en el grafo. La ruta más corta probablemente pase por ciertos nodos intermedios. La **betweenness centrality** cuenta cuántas de estas "rutas más cortas" entre todos los pares de nodos del grafo pasan por cada nodo.

Un valor alto significa que este usuario es un **puente** entre distintos grupos. Si desapareciera, muchas rutas de comunicación dejarían de existir. En el contexto de carding, esto puede indicar un intermediario o un actor de confianza que conecta distintas comunidades.

In [ ]:
# Calculamos métricas sobre la componente gigante
# Nota: betweenness_centrality puede tardar varios minutos en grafos grandes
print("Calculando degree centrality...")
degree_centrality = nx.degree_centrality(G_giant)

print("Calculando betweenness centrality (puede tardar)...")
# k=500 usa un muestreo aproximado — mucho más rápido para grafos grandes
betweenness = nx.betweenness_centrality(G_giant, k=min(500, len(G_giant)), normalized=True)

print("Calculando clustering coefficient...")
clustering = nx.clustering(G_giant)

# Combinar en un DataFrame
centrality_df = pd.DataFrame({
    "userid": list(degree_centrality.keys()),
    "degree_centrality": list(degree_centrality.values()),
    "betweenness_centrality": [betweenness.get(n, 0) for n in degree_centrality.keys()],
    "clustering": [clustering.get(n, 0) for n in degree_centrality.keys()],
})

# Enriquecer con metadata de usuario
centrality_df = centrality_df.merge(
    users[["userid", "forum", "username", "posts"]].drop_duplicates(subset="userid"),
    on="userid", how="left"
)

print(f"\nTop 15 por betweenness centrality (posibles intermediarios/puentes):")
centrality_df.sort_values("betweenness_centrality", ascending=False).head(15)[
    ["username", "forum", "degree_centrality", "betweenness_centrality", "clustering", "posts"]
]

In [ ]:
# Guardar para el notebook de síntesis
centrality_df.to_parquet(RESULTS_DIR / "02_centrality.parquet", index=False)

# Visualización: degree vs betweenness
fig = px.scatter(
    centrality_df.nlargest(200, "degree_centrality"),
    x="degree_centrality",
    y="betweenness_centrality",
    color="forum",
    hover_data=["username", "posts"],
    title="Centralidad de usuarios — grado vs betweenness (top 200)",
    labels={
        "degree_centrality": "Degree centrality (conectividad directa)",
        "betweenness_centrality": "Betweenness centrality (rol de puente)"
    },
    size="posts", size_max=20,
)
fig.write_html(RESULTS_DIR / "02_centrality_scatter.html")
fig.show()

print("\nInterpretación:")
print("  - Cuadrante superior derecha: alto degree Y alta betweenness → hubs puente (intermediarios clave)")
print("  - Alto degree, baja betweenness → usuarios populares dentro de un cluster")
print("  - Baja degree, alta betweenness → puentes entre clusters pequeños (interesante)")

### ¿Por qué filtrar aristas de un solo thread compartido antes de correr Louvain?

Al construir el grafo (sección 2) ya aplicamos `MIN_SHARED_THREADS = 2`: solo mantenemos una
arista entre dos usuarios si coincidieron en **al menos 2 threads distintos**, no solo uno.

**¿Por qué importa esto para Louvain?** Louvain agrupa nodos según la densidad de sus
conexiones. Si dejáramos aristas de "1 thread compartido" —dos usuarios que por pura
coincidencia postearon una vez en el mismo hilo masivo (un thread con cientos de
participantes)— Louvain podría fusionar comunidades completamente ajenas en una sola,
simplemente porque comparten ese hilo de alto tráfico. Filtrar esas coincidencias triviales
antes de correr el algoritmo evita comunidades falsas que no reflejan ninguna relación real.


In [ ]:
# Efecto del filtro de aristas débiles (1 thread compartido) sobre el grafo
total_edges = len(edges_df)
strong_edges_count = len(strong_edges)
removed = total_edges - strong_edges_count

print(f"Aristas totales (≥1 thread compartido): {total_edges:,}")
print(f"Aristas fuertes (≥{MIN_SHARED_THREADS} threads compartidos, las que usamos): {strong_edges_count:,}")
print(f"Aristas descartadas por ruido de 1 thread: {removed:,} ({100*removed/total_edges:.1f}%)")


## 4. Detección de comunidades con Louvain

### ¿Qué es la detección de comunidades?

En una red de usuarios, existen grupos naturales donde los nodos están más densamente conectados entre sí que con el resto de la red. Estos grupos se llaman **comunidades** o **clusters**.

En el contexto de carding, una comunidad puede representar:
- Un círculo de confianza (usuarios que siempre operan juntos)
- Una especialización temática (vendedores de un tipo específico de producto)
- Una procedencia geográfica o lingüística

### ¿Qué es el algoritmo de Louvain?

Louvain es un algoritmo de optimización que intenta maximizar la **modularidad** de la red: una medida de qué tan bien separadas están las comunidades entre sí. 

No necesitamos especificar cuántas comunidades queremos — el algoritmo las encuentra solo.

### ¿Por qué Louvain y no k-means u otro clustering?

k-means necesita que le digamos de antemano cuántos clusters (`K`) queremos, y asume que
los puntos viven en un espacio donde "distancia euclidiana" tiene sentido — aquí no tenemos
coordenadas, tenemos un grafo de conexiones. Forzar esos vectores a k-means significaría
inventar una representación geométrica solo para poder clusterizar, perdiendo la estructura
real de quién-conecta-con-quién.

Louvain, en cambio, trabaja directamente sobre el grafo optimizando **modularidad**: no fija
`K` (encuentra el número de comunidades que mejor explica la estructura observada), detecta
comunidades naturales (grupos más densamente conectados internamente que con el resto), y
escala razonablemente bien incluso en grafos de decenas de miles de nodos como el nuestro
(11,113 nodos en la componente gigante) — una optimización jerárquica local, no una búsqueda
exhaustiva.


In [ ]:
try:
    import community as community_louvain
    
    # Detectar comunidades en la componente gigante
    partition = community_louvain.best_partition(G_giant, weight="weight", random_state=42)
    
    n_communities = len(set(partition.values()))
    modularity = community_louvain.modularity(partition, G_giant)
    
    print(f"Comunidades detectadas: {n_communities}")
    print(f"Modularidad: {modularity:.3f}")
    print("  (Modularidad > 0.3 indica estructura de comunidad significativa)")
    
    # Tamaño de cada comunidad
    community_sizes = Counter(partition.values())
    size_series = pd.Series(community_sizes).sort_values(ascending=False)
    print(f"\nTamaño de comunidades (top 10):")
    print(size_series.head(10).to_string())
    
    # Añadir comunidad al DataFrame de centralidad
    centrality_df["community"] = centrality_df["userid"].map(partition)
    
except ImportError:
    print("python-louvain no está instalado.")
    print("Instalá con: pip install python-louvain")
    print("Alternativa: NetworkX tiene greedy_modularity_communities()")
    
    # Alternativa con NetworkX puro (más lento pero sin dependencias extra)
    communities = nx.community.greedy_modularity_communities(G_giant)
    partition = {}
    for i, comm in enumerate(communities):
        for node in comm:
            partition[node] = i
    
    n_communities = len(communities)
    print(f"Comunidades detectadas (greedy): {n_communities}")
    centrality_df["community"] = centrality_df["userid"].map(partition)

In [ ]:
# ¿Qué foros dominan en cada comunidad?
# Si una comunidad tiene usuarios de un solo foro, es un cluster interno.
# Si tiene usuarios de múltiples foros, puede ser una red cross-foro.

if "community" in centrality_df.columns:
    community_composition = (
        centrality_df
        .groupby(["community", "forum"])
        .size()
        .reset_index(name="n_users")
        .sort_values(["community", "n_users"], ascending=[True, False])
    )
    
    # Mostrar composición de las 5 comunidades más grandes
    top_communities = size_series.head(5).index.tolist() if 'size_series' in dir() else list(range(5))
    print("Composición de las 5 comunidades más grandes:")
    print(
        community_composition[
            community_composition["community"].isin(top_communities)
        ].to_string(index=False)
    )

**Nota al margen — comunidad mixta (varios foros):** si alguna de las comunidades grandes
mezcla usuarios de foros distintos, es candidata a ser una red cross-foro (mismos actores,
distintas plataformas). Aquí solo lo señalamos de pasada — la atribución de identidad
cross-foro en profundidad (multi-señal, persistencia temporal) es el hilo conductor del caso
HackingForums, no lo repetimos aquí.


## 5. Visualización interactiva de la red

Visualizamos los top 150 nodos por centralidad para que el grafo sea legible. Un grafo con miles de nodos es visualmente inutilizable.

**¿Qué es el layout de Fruchterman-Reingold?**  
Es un algoritmo de posicionamiento de nodos basado en fuerzas: los nodos conectados se atraen, los no conectados se repelen. El resultado es que los clusters se separan visualmente de forma natural.

In [ ]:
# Seleccionar top 150 nodos por degree
top_nodes = centrality_df.nlargest(150, "degree_centrality")["userid"].tolist()
G_viz = G_giant.subgraph(top_nodes).copy()

# Calcular layout
pos = nx.spring_layout(G_viz, k=2, seed=42)

# Preparar datos para Plotly
node_x, node_y, node_labels, node_colors, node_sizes = [], [], [], [], []

uid_to_meta = centrality_df.set_index("userid")

for node in G_viz.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    if node in uid_to_meta.index:
        meta = uid_to_meta.loc[node]
        name = meta["username"] if pd.notna(meta.get("username")) else str(node)
        node_labels.append(f"{name}<br>Forum: {meta.get('forum', '?')}<br>Posts: {meta.get('posts', '?')}")
        node_sizes.append(max(8, float(meta["degree_centrality"]) * 100))
        node_colors.append(meta.get("community", 0) if "community" in uid_to_meta.columns else 0)
    else:
        node_labels.append(str(node))
        node_sizes.append(8)
        node_colors.append(0)

# Aristas
edge_x, edge_y = [], []
for u, v in G_viz.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode="lines",
    line=dict(width=0.5, color="#aaa"),
    hoverinfo="none"
))
fig.add_trace(go.Scatter(
    x=node_x, y=node_y, mode="markers",
    marker=dict(size=node_sizes, color=node_colors, colorscale="Turbo", showscale=True),
    text=node_labels,
    hoverinfo="text"
))
fig.update_layout(
    title="Red de co-participación — Top 150 usuarios (Fruchterman-Reingold)",
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=700
)
fig.write_html(RESULTS_DIR / "02_network_viz.html")
fig.show()
print("\nViz interactiva guardada en results/02_network_viz.html")

## 6. Análisis temporal

### ¿Cuándo estaban activos estos foros?

El análisis temporal nos permite ver si hay patrones de actividad — estacionalidad, picos relacionados con eventos externos (grandes brechas de datos, operaciones policiales), o patrones horarios que ayuden a inferir timezones.

In [ ]:
# Actividad mensual por foro
posts_dated = posts[posts["dateline_valid"] == True].copy()

monthly = (
    posts_dated
    .assign(month=posts_dated["dateline"].dt.to_period("M"))
    .groupby(["month", "forum"])
    .size()
    .reset_index(name="posts")
)
monthly["month"] = monthly["month"].dt.to_timestamp()

fig = px.area(
    monthly, x="month", y="posts", color="forum",
    title="Actividad mensual por foro (posts publicados)",
    labels={"month": "Mes", "posts": "Posts", "forum": "Foro"}
)
fig.write_html(RESULTS_DIR / "02_temporal_activity.html")
fig.show()

In [ ]:
# Patrón horario: ¿a qué hora del día se publica más?
# Esto nos da una pista de la distribución geográfica de los usuarios
# (UTC, así que hay que interpretar con cuidado)

posts_dated["hour_utc"] = posts_dated["dateline"].dt.hour

hourly = posts_dated.groupby(["hour_utc", "forum"]).size().reset_index(name="posts")

fig = px.line(
    hourly, x="hour_utc", y="posts", color="forum",
    title="Distribución horaria de posts (UTC)",
    labels={"hour_utc": "Hora UTC", "posts": "Posts"},
)
fig.update_xaxes(tickvals=list(range(0, 24, 2)))
fig.write_html(RESULTS_DIR / "02_hourly_pattern.html")
fig.show()

print("Interpretación:")
print("  El pico de actividad en hora UTC nos dice en qué zona horaria están activos.")
print("  Pico a las 10-14 UTC → Europa del Este (UTC+3 = Moscú/San Petersburgo)")
print("  Pico a las 18-22 UTC → Américas (UTC-5 a UTC-8)")

## 7. TF-IDF: términos discriminantes por subforo

### ¿Qué es TF-IDF?

**TF-IDF** significa *Term Frequency — Inverse Document Frequency*. Es una medida que nos dice qué tan **característico** es un término para un documento (en nuestro caso, un subforo) en comparación con todos los demás documentos.

- **TF (frecuencia de término)**: qué tan frecuente es la palabra en este subforo. "Dumps" puede aparecer mucho en el subforo de dumps.
- **IDF (frecuencia inversa de documento)**: qué tan rara es la palabra en el conjunto completo. "El" o "de" aparecen en todos los subforos, así que no dicen nada específico.

**TF-IDF alto** = la palabra es frecuente en ESTE subforo pero rara en los otros → es una palabra discriminante y característica.

Esto nos ayuda a responder: ¿de qué habla cada subforo en realidad?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

if "text_clean" in posts.columns and "forumid" in posts.columns:
    # Agrupar texto por subforo
    subforum_corpus = (
        posts[posts["text_valid"] == True]
        .groupby(["forum", "forumid"])["text_clean"]
        .apply(lambda texts: " ".join(texts.dropna()))
        .reset_index(name="text")
    )
    
    # TF-IDF sobre los subforos (cada subforo = un "documento")
    vectorizer = TfidfVectorizer(
        max_features=5000,
        min_df=2,          # La palabra debe aparecer en al menos 2 subforos
        stop_words="english",  # Eliminamos palabras vacías en inglés
        ngram_range=(1, 2) # Incluimos bigramas ("credit card", "dump shop")
    )
    
    tfidf_matrix = vectorizer.fit_transform(subforum_corpus["text"])
    feature_names = vectorizer.get_feature_names_out()
    
    # Top términos por subforo
    print("Términos más característicos por subforo (top 10 por TF-IDF):")
    print("=" * 60)
    
    for i, row in subforum_corpus.iterrows():
        top_indices = tfidf_matrix[i].toarray()[0].argsort()[-10:][::-1]
        top_terms = [feature_names[j] for j in top_indices]
        print(f"\n[{row['forum']} / subforo {row['forumid']}]")
        print("  " + ", ".join(top_terms))
else:
    print("Nota: Para el análisis de TF-IDF por subforo necesitamos la columna 'forumid' en posts.")
    print("Esta columna se añade en el notebook 01 cuando se hace el join con la tabla threads.")
    print("\nAlternativa: TF-IDF por foro completo (más general)")
    
    if "text_clean" in posts.columns:
        forum_corpus = (
            posts[posts.get("text_valid", pd.Series(True, index=posts.index)) == True]
            .groupby("forum")["text_clean"]
            .apply(lambda texts: " ".join(texts.dropna()))
        )
        
        vectorizer = TfidfVectorizer(max_features=2000, stop_words="english", ngram_range=(1, 2))
        tfidf_matrix = vectorizer.fit_transform(forum_corpus)
        feature_names = vectorizer.get_feature_names_out()
        
        for i, forum_name in enumerate(forum_corpus.index):
            top_indices = tfidf_matrix[i].toarray()[0].argsort()[-10:][::-1]
            top_terms = [feature_names[j] for j in top_indices]
            print(f"\n[{forum_name}]")
            print("  " + ", ".join(top_terms))

## 8. Resumen estructural

Consolidamos los hallazgos de este notebook para el notebook de síntesis.

In [ ]:
# Guardar ranking de centralidad para notebook 04
centrality_df.to_parquet(RESULTS_DIR / "02_centrality.parquet", index=False)

print("RESUMEN DEL ANÁLISIS ESTRUCTURAL")
print("=" * 50)
print(f"  Nodos en el grafo: {G.number_of_nodes():,}")
print(f"  Aristas (conexiones fuertes ≥{MIN_SHARED_THREADS} threads): {G.number_of_edges():,}")
if 'n_communities' in dir():
    print(f"  Comunidades detectadas: {n_communities}")

print(f"\nTop 5 por betweenness centrality (probables intermediarios):")
top5 = centrality_df.nlargest(5, "betweenness_centrality")[["username", "forum", "betweenness_centrality"]]
print(top5.to_string(index=False))

print("\n→ Siguiente paso: 03_analisis_semantico.ipynb")